# Wall design — quick path

Strict step-by-step. We always **plot the original section** before
any design call, and **plot the final section** after, so we can
verify visually that the rebar layout is clean (no stacked bars).

Convention: the wall starts at its web. When there is no BE, the
distributed reinforcement runs the full `lw`. When a BE is added,
web bars inside the new BE rectangle are dropped and the BE bars
take their place.

## 0. Path setup

This notebook lives inside `design/`. We add the repo root to
`sys.path` so `from design import ...` resolves regardless of
where Jupyter was launched.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'design' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('repo root:', repo_root)

In [ ]:
import matplotlib.pyplot as plt
from design import Concrete, Steel, Bar, Wall
from design.walls import WallDemands

DISPLAY_UNITS = 5    # kN_mm_C — keep moments in kN.mm for readability

## 1. Build the section — no boundary elements yet

Web bars run edge to edge of the wall length `lw`. We will see
later that, when the design forces a BE, these bars get trimmed
at the BE boundary automatically.

In [ ]:
concrete = Concrete(fc=35)
steel    = Steel(fy=420, grade=60)

# Wall.rectangular() — barbell wall con BE column-style.
# Internal units: mm, MPa, kN, kN·m.
# §18.10.6.4(b): n_x_be >= 3 obligatorio (default = 3).
wall = Wall.rectangular(
    lw=4000, tw=300, hw=30_000, lu=3_000,
    concrete=concrete, steel=steel,
    # web vertical bars
    n_y_web=14, db_web=12, cover=50,
    # BARBELL boundary elements (BE thickness > tw)
    be_top_length=600, be_top_thickness=500,
    be_bot_length=600, be_bot_thickness=500,
    n_x_be=4, n_y_be=5, db_be=22,
    seismic=True, label='W-1',
    units=DISPLAY_UNITS,
)
section = wall.section
print(section)
print('Ag =', section.gross_area(), 'mm^2  |  n bars =', sum(len(g.bars) for g in section.rebar.groups))

### Plot the ORIGINAL section (visual sanity check)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 8))
wall.plot.section(ax=ax)
plt.show()

## 2. Capacity of the original geometry

In [ ]:
wall.run()
wall.summary()

### Interaction diagram (in-plane) — original geometry

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
wall.plot.pm(plane='in-plane', ax=ax)
plt.show()

### PMM volume — original geometry

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
wall.plot.pmm_volume(ax=ax, n_Pu=20)
plt.show()

## 3. Define demands

`delta_u` and `sigma_max` come from the analysis; they control
the §18.10.6 BE-required checks.

In [ ]:
demands = WallDemands(
    Pu=2000.0, Mu=8000.0, Vu=1500.0,
    delta_u=180.0, sigma_max=8.0,
)
chk = wall.check(demands)
print(f'ratio PM        = {chk.ratio_pm:.3f}')
print(f'ratio shear     = {chk.ratio_shear:.3f}')
print(f'BE disp/stress  = {chk.be_required_disp} / {chk.be_required_stress}')
print(f'c at demand     = {chk.c_at_demand:.0f} mm')
print(f'passed          = {chk.passed}')

## 4. Design with auto_update — BE is proposed and added

If §18.10.6 triggers, `wall.design()` proposes BE geometry,
rebuilds the section, and re-runs the PMM. Iterates until the
BE length stabilises.

In [ ]:
results = wall.design(demands, auto_update=True)
print(f'iterations: {results.iterations}, converged: {results.converged}')
for i, w in enumerate(results.history):
    print(f'  iter {i}: BE top = {w.section.be_length_top:>4.0f} mm, BE bot = {w.section.be_length_bot:>4.0f} mm')

## 5. Plot the FINAL section — verify no stacked bars

In [ ]:
final = results.final_wall
fig, ax = plt.subplots(figsize=(3, 8))
final.plot.section(ax=ax)
plt.show()

n_total = sum(len(g.bars) for g in final.section.rebar.groups)
n_web   = sum(len(g.bars) for g in final.section.web_rebar().groups)
n_top   = sum(len(g.bars) for g in (final.section.be_top_rebar() or RebarLayout()).groups)
n_bot   = sum(len(g.bars) for g in (final.section.be_bot_rebar() or RebarLayout()).groups)
print(f'bars total = {n_total}  ({n_web} web + {n_top} BE top + {n_bot} BE bot)')

## 6. Compare interaction diagrams: original vs final

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
results.original_wall.plot.pm(plane='in-plane', ax=axes[0])
axes[0].set_title('ORIGINAL — no BE')
final.plot.pm(plane='in-plane', ax=axes[1])
axes[1].set_title('FINAL — with BE')
plt.tight_layout()
plt.show()

## 7. Inspect the BE as a Column

In [ ]:
if final.has_boundary_elements:
    be = final.be_top
    print(be)
    fig, ax = plt.subplots(figsize=(5, 5))
    be.plot.section(ax=ax)
    plt.show()
else:
    print('No BE was added — demands did not trigger §18.10.6.')

## 8. Envelope summary

In [ ]:
env = results.envelope
print(f'rho_t required     = {env.rho_t_required:.4f}')
print(f'BE required        = {env.be_required}')
print(f'BE length proposed = {env.be_length_proposed:.0f} mm')
print(f'BE thick proposed  = {env.be_thickness_proposed:.0f} mm')
print(f'Mpr in-plane       = {env.Mpr_in_plane:.0f} kN.m')
print(f'Ve capacity        = {env.Ve_capacity:.0f} kN')
print(f'phi_Vn             = {env.phi_Vn:.0f} kN  (cap = {env.phi_Vn_max:.0f} kN)')

## 9. Reinforcement quantities — optimize sweep

`wall.run_optimize(demands)` enumerates feasible combinations of

  * web distributed bars `(db_web, spacing, layers)`,
  * BE longitudinal bars `(db_be, n_y_be)` when the wall has BEs,

and ranks them ascending by total steel mass (kg/m³ of
concrete). The output mirrors the tables produced by
`beam.run_optimize()` and `column.run_optimize()`.

The `baseline` row is the detailing that `wall.design()` would
have chosen. Rows below it in the ranking are lighter
alternatives that still satisfy ρ_min and the spacing cap.
Anything tagged `infeasible` violates a code limit and is
hidden by default (pass `include_infeasible=True` to see it).

In [ ]:
from design.walls.optimize import format_optimize_table

# Use the wall as it stands AFTER design() — i.e. the final
# wall with the BE that auto_update added. This gives the most
# realistic optimize sweep.
alts = final.run_optimize(demands)
print(f'Alternatives explored: {len(alts)}')
print(f'Lightest rho_total  : {alts[0].rho_total:.1f} kg/m^3')
print()
print(format_optimize_table(alts, top_n=10))